# Standalone XLM-R and mBERT Baselines for Thai Emotion Datasets

Notebook เดียวสำหรับรัน baseline **2 โมเดลแบบ standalone**
โดย **ไม่มี X-Transformers / ไม่มี EAQD / ไม่มี Optuna**

Models:
1. `xlm-roberta-base` — XLM-R
2. `bert-base-multilingual-cased` — mBERT

Datasets:
1. **SIED-Thai original / unbalanced**
   - Happiness
   - Sadness
   - Anger
   - Disgust
   - Surprise
   - Fear
2. **Thai-Depression**
   - no_depression
   - depression

## วิธีทำงาน
กด **Run All ครั้งเดียว** แล้ว notebook จะรันตามลำดับ:

1. XLM-R → SIED-Thai
2. mBERT → SIED-Thai
3. XLM-R → Thai-Depression
4. mBERT → Thai-Depression

แต่ละโมเดลเป็น:
**Pretrained Encoder → built-in Classification Head**

ไม่มี X-Transformers ต่อท้าย

## Output
- Overall Accuracy / Micro-F1 / Macro-F1
- Precision / Recall / F1-score / Support รายหมวด
- Hamming Loss สำหรับ SIED-Thai
- threshold ราย label สำหรับ SIED-Thai
- ตารางเปรียบเทียบ XLM-R vs mBERT
- CSV พร้อมนำไปใส่ paper

In [ ]:
!pip install -q "transformers>=4.44,<5" sentencepiece scikit-learn pandas tqdm

In [ ]:
import os
import glob
import gc
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    classification_report
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("DEVICE:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
DEVICE: cuda
GPU: Tesla T4


In [ ]:
# =========================================================
# CONFIG
# =========================================================

MODELS = {
    "XLM-R": "xlm-roberta-base",
    "mBERT": "bert-base-multilingual-cased",
}

SEED = 42

MAX_LEN = 128

# Full fine-tuning ของ pretrained encoder ใช้ batch จริง 16
# + gradient accumulation 4 = effective batch size 64
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 4
EFFECTIVE_BATCH_SIZE = TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS

# Standalone pretrained Transformer baseline
# ค่า default 5 epochs เพื่อประหยัดเวลา
EPOCHS = 5

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

# เลือกว่าจะรัน dataset ใด
RUN_SIED = True
RUN_THAI_DEPRESSION = True

# ถ้า True จะ save checkpoint ของแต่ละ baseline ด้วย
SAVE_MODELS = False

SIED_LABELS = [
    "Happiness",
    "Sadness",
    "Anger",
    "Disgust",
    "Surprise",
    "Fear",
]

DEPRESSION_LABELS = [
    "no_depression",
    "depression",
]

OUTPUT_DIR = (
    "/kaggle/working"
    if os.path.exists("/kaggle/working")
    else "/content"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Effective batch size:", EFFECTIVE_BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Output:", OUTPUT_DIR)

Effective batch size: 64
Epochs: 5
Output: /content


In [ ]:
# =========================================================
# REPRODUCIBILITY
# =========================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

In [ ]:
# =========================================================
# FIND DATASETS AUTOMATICALLY
# =========================================================

def find_file(filename, preferred_keyword=None):

    candidates = [
        f"/content/{filename}",
        f"/mnt/data/{filename}",
    ]

    for path in candidates:
        if os.path.exists(path):
            return path

    hits = glob.glob(
        f"/kaggle/input/**/{filename}",
        recursive=True
    )

    if preferred_keyword:
        preferred = [
            p for p in hits
            if preferred_keyword.lower() in p.lower()
        ]
        if preferred:
            return preferred[0]

    if hits:
        return hits[0]

    return None


SIED_PATH = find_file(
    "SIED-Thai.csv",
    "sied"
)

DEP_TRAIN_PATH = find_file(
    "train.json",
    "depression"
)

DEP_VALID_PATH = find_file(
    "valid.json",
    "depression"
)

DEP_TEST_PATH = find_file(
    "test.json",
    "depression"
)

print("SIED:", SIED_PATH)
print("Thai Depression train:", DEP_TRAIN_PATH)
print("Thai Depression valid:", DEP_VALID_PATH)
print("Thai Depression test :", DEP_TEST_PATH)

if RUN_SIED and SIED_PATH is None:
    raise FileNotFoundError(
        "ไม่พบ SIED-Thai.csv กรุณา Add dataset ใน Kaggle "
        "หรือ upload เป็น /content/SIED-Thai.csv"
    )

if RUN_THAI_DEPRESSION and any(
    p is None
    for p in [DEP_TRAIN_PATH, DEP_VALID_PATH, DEP_TEST_PATH]
):
    raise FileNotFoundError(
        "ไม่พบ train.json / valid.json / test.json ของ Thai-Depression"
    )

SIED: /content/SIED-Thai.csv
Thai Depression train: /content/train.json
Thai Depression valid: /content/valid.json
Thai Depression test : /content/test.json


In [ ]:
# =========================================================
# LOAD SIED-THAI ORIGINAL / UNBALANCED
# ใช้ fixed 70/15/15 split ด้วย seed=42
# เพื่อให้เปรียบเทียบกับ current frozen-model experiment ได้
# =========================================================

def load_sied():

    df = pd.read_csv(SIED_PATH)

    required = ["Tweets"] + SIED_LABELS

    missing = [
        c for c in required
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"SIED-Thai missing columns: {missing}"
        )

    df = df[required].copy()

    df["Tweets"] = (
        df["Tweets"]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    for col in SIED_LABELS:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        ).fillna(0).astype(int)

    train_df, temp_df = train_test_split(
        df,
        test_size=0.30,
        random_state=SEED,
        shuffle=True
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=SEED,
        shuffle=True
    )

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )


if RUN_SIED:

    sied_train_df, sied_val_df, sied_test_df = load_sied()

    print("SIED train:", len(sied_train_df))
    print("SIED valid:", len(sied_val_df))
    print("SIED test :", len(sied_test_df))

    print("\nSIED train label distribution:")
    print(sied_train_df[SIED_LABELS].sum())

SIED train: 1680
SIED valid: 360
SIED test : 360

SIED train label distribution:
Happiness     122
Sadness      1315
Anger         227
Disgust       324
Surprise      117
Fear          222
dtype: int64


In [ ]:
# =========================================================
# LOAD THAI-DEPRESSION OFFICIAL SPLITS
# =========================================================

def load_depression_json(path):

    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    texts = []
    labels = []

    for item in raw:

        if not isinstance(item, (list, tuple)):
            continue

        if len(item) != 2:
            continue

        text, label = item

        if not isinstance(text, str):
            continue

        text = (
            text
            .replace("---- “ ----", " ")
            .replace("\n", " ")
        )

        text = " ".join(text.split()).strip()

        if len(text) < 2:
            continue

        if label == "no_depression":
            label_id = 0

        elif label == "depression":
            label_id = 1

        else:
            continue

        texts.append(text)
        labels.append(label_id)

    return pd.DataFrame({
        "text": texts,
        "label": labels
    })


if RUN_THAI_DEPRESSION:

    dep_train_df = load_depression_json(
        DEP_TRAIN_PATH
    )

    dep_val_df = load_depression_json(
        DEP_VALID_PATH
    )

    dep_test_df = load_depression_json(
        DEP_TEST_PATH
    )

    print("Thai Depression train:", len(dep_train_df))
    print("Thai Depression valid:", len(dep_val_df))
    print("Thai Depression test :", len(dep_test_df))

    print("\nTrain class distribution:")
    print(
        dep_train_df["label"]
        .value_counts()
        .sort_index()
    )

Thai Depression train: 25004
Thai Depression valid: 3335
Thai Depression test : 5004

Train class distribution:
label
0    12239
1    12765
Name: count, dtype: int64


In [ ]:
# =========================================================
# DATASET CLASSES
# =========================================================

class SIEDDataset(Dataset):

    def __init__(
        self,
        frame,
        tokenizer
    ):
        self.texts = frame["Tweets"].tolist()
        self.targets = (
            frame[SIED_LABELS]
            .values
            .astype(np.float32)
        )
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        return {
            "input_ids":
                enc["input_ids"].squeeze(0),

            "attention_mask":
                enc["attention_mask"].squeeze(0),

            "labels":
                torch.tensor(
                    self.targets[idx],
                    dtype=torch.float32
                )
        }


class DepressionDataset(Dataset):

    def __init__(
        self,
        frame,
        tokenizer
    ):
        self.texts = frame["text"].tolist()
        self.targets = (
            frame["label"]
            .values
            .astype(np.int64)
        )
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        return {
            "input_ids":
                enc["input_ids"].squeeze(0),

            "attention_mask":
                enc["attention_mask"].squeeze(0),

            "labels":
                torch.tensor(
                    self.targets[idx],
                    dtype=torch.long
                )
        }

In [ ]:
# =========================================================
# DATALOADER HELPERS
# =========================================================

def make_sied_loaders(
    tokenizer,
    seed=42
):
    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        SIEDDataset(sied_train_df, tokenizer),
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        generator=generator
    )

    val_loader = DataLoader(
        SIEDDataset(sied_val_df, tokenizer),
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False
    )

    test_loader = DataLoader(
        SIEDDataset(sied_test_df, tokenizer),
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False
    )

    return train_loader, val_loader, test_loader


def make_dep_loaders(
    tokenizer,
    seed=42
):
    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        DepressionDataset(dep_train_df, tokenizer),
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        generator=generator
    )

    val_loader = DataLoader(
        DepressionDataset(dep_val_df, tokenizer),
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False
    )

    test_loader = DataLoader(
        DepressionDataset(dep_test_df, tokenizer),
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False
    )

    return train_loader, val_loader, test_loader

In [ ]:
# =========================================================
# SIED METRICS + PER-LABEL THRESHOLD TUNING
# =========================================================

@torch.no_grad()
def predict_multilabel(
    model,
    loader
):

    model.eval()

    probs_all = []
    labels_all = []

    for batch in loader:

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).logits

        probs = torch.sigmoid(logits)

        probs_all.append(
            probs.cpu().numpy()
        )

        labels_all.append(
            batch["labels"].numpy()
        )

    return (
        np.vstack(labels_all),
        np.vstack(probs_all)
    )


def tune_multilabel_thresholds(
    y_true,
    y_prob
):

    thresholds = []

    candidates = np.arange(
        0.20,
        0.805,
        0.01
    )

    for i in range(
        len(SIED_LABELS)
    ):

        best_thr = 0.50
        best_f1 = -1.0
        best_precision = -1.0

        for thr in candidates:

            pred = (
                y_prob[:, i] >= thr
            ).astype(int)

            f1 = f1_score(
                y_true[:, i],
                pred,
                zero_division=0
            )

            precision = precision_score(
                y_true[:, i],
                pred,
                zero_division=0
            )

            if (
                f1 > best_f1
                or (
                    np.isclose(f1, best_f1)
                    and precision > best_precision
                )
            ):
                best_f1 = f1
                best_precision = precision
                best_thr = thr

        thresholds.append(best_thr)

    return np.array(thresholds)


def multilabel_metrics(
    y_true,
    y_prob,
    thresholds
):

    y_pred = (
        y_prob >= thresholds
    ).astype(int)

    overall = {
        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "micro_precision":
            precision_score(
                y_true,
                y_pred,
                average="micro",
                zero_division=0
            ),

        "micro_recall":
            recall_score(
                y_true,
                y_pred,
                average="micro",
                zero_division=0
            ),

        "micro_f1":
            f1_score(
                y_true,
                y_pred,
                average="micro",
                zero_division=0
            ),

        "macro_precision":
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "macro_recall":
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "weighted_f1":
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0
            ),

        "hamming_loss":
            hamming_loss(
                y_true,
                y_pred
            )
    }

    p = precision_score(
        y_true,
        y_pred,
        average=None,
        zero_division=0
    )

    r = recall_score(
        y_true,
        y_pred,
        average=None,
        zero_division=0
    )

    f = f1_score(
        y_true,
        y_pred,
        average=None,
        zero_division=0
    )

    support = y_true.sum(
        axis=0
    ).astype(int)

    per_label = pd.DataFrame({
        "label": SIED_LABELS,
        "precision": p,
        "recall": r,
        "f1": f,
        "support": support,
        "threshold": thresholds,
    })

    return overall, per_label

In [ ]:
# =========================================================
# THAI-DEPRESSION METRICS
# =========================================================

@torch.no_grad()
def predict_single_label(
    model,
    loader
):

    model.eval()

    y_true = []
    y_pred = []

    for batch in loader:

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).logits

        pred = torch.argmax(
            logits,
            dim=1
        )

        y_true.extend(
            batch["labels"]
            .cpu()
            .numpy()
        )

        y_pred.extend(
            pred
            .cpu()
            .numpy()
        )

    return (
        np.array(y_true),
        np.array(y_pred)
    )


def single_label_metrics(
    y_true,
    y_pred
):

    overall = {
        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "micro_precision":
            precision_score(
                y_true,
                y_pred,
                average="micro",
                zero_division=0
            ),

        "micro_recall":
            recall_score(
                y_true,
                y_pred,
                average="micro",
                zero_division=0
            ),

        "micro_f1":
            f1_score(
                y_true,
                y_pred,
                average="micro",
                zero_division=0
            ),

        "macro_precision":
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "macro_recall":
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "weighted_f1":
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0
            )
    }

    p = precision_score(
        y_true,
        y_pred,
        labels=[0, 1],
        average=None,
        zero_division=0
    )

    r = recall_score(
        y_true,
        y_pred,
        labels=[0, 1],
        average=None,
        zero_division=0
    )

    f = f1_score(
        y_true,
        y_pred,
        labels=[0, 1],
        average=None,
        zero_division=0
    )

    support = np.array([
        (y_true == 0).sum(),
        (y_true == 1).sum()
    ])

    per_label = pd.DataFrame({
        "label": DEPRESSION_LABELS,
        "precision": p,
        "recall": r,
        "f1": f,
        "support": support
    })

    return overall, per_label

In [ ]:
# =========================================================
# TRAIN STANDALONE MODEL
# =========================================================

def train_standalone(
    model_name,
    dataset_name
):

    seed_everything(SEED)

    tokenizer = AutoTokenizer.from_pretrained(
        model_name
    )

    if dataset_name == "SIED-Thai":

        num_labels = len(SIED_LABELS)

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
            problem_type="multi_label_classification"
        ).to(DEVICE)

        train_loader, val_loader, test_loader = make_sied_loaders(
            tokenizer,
            SEED
        )

        positives = (
            sied_train_df[SIED_LABELS]
            .sum()
            .values
            .astype(np.float32)
        )

        negatives = (
            len(sied_train_df)
            - positives
        )

        pos_weight = torch.tensor(
            negatives / (positives + 1e-6),
            dtype=torch.float32,
            device=DEVICE
        )

        criterion = nn.BCEWithLogitsLoss(
            pos_weight=pos_weight
        )

    elif dataset_name == "Thai-Depression":

        num_labels = 2

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_labels
        ).to(DEVICE)

        train_loader, val_loader, test_loader = make_dep_loaders(
            tokenizer,
            SEED
        )

        counts = (
            dep_train_df["label"]
            .value_counts()
            .sort_index()
            .values
            .astype(np.float32)
        )

        class_weights = (
            1.0 / counts
        )

        class_weights = (
            class_weights
            / class_weights.sum()
            * len(class_weights)
        )

        class_weights = torch.tensor(
            class_weights,
            dtype=torch.float32,
            device=DEVICE
        )

        criterion = nn.CrossEntropyLoss(
            weight=class_weights
        )

    else:
        raise ValueError(dataset_name)


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    steps_per_epoch = int(
        np.ceil(
            len(train_loader)
            / GRAD_ACCUM_STEPS
        )
    )

    total_steps = (
        steps_per_epoch
        * EPOCHS
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(
            WARMUP_RATIO * total_steps
        ),
        num_training_steps=total_steps
    )

    use_amp = (
        DEVICE.type == "cuda"
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=use_amp
    )

    best_val_macro_f1 = -1.0
    best_state = None

    for epoch in range(EPOCHS):

        model.train()

        optimizer.zero_grad()

        total_loss = 0.0

        progress = tqdm(
            train_loader,
            desc=f"{dataset_name} | Epoch {epoch+1}/{EPOCHS}",
            leave=False
        )

        for step, batch in enumerate(
            progress,
            start=1
        ):

            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            with torch.cuda.amp.autocast(
                enabled=use_amp
            ):

                logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                ).logits

                loss = criterion(
                    logits,
                    labels
                )

                loss = (
                    loss
                    / GRAD_ACCUM_STEPS
                )

            scaler.scale(
                loss
            ).backward()

            total_loss += (
                loss.item()
                * GRAD_ACCUM_STEPS
            )

            if (
                step % GRAD_ACCUM_STEPS == 0
                or step == len(train_loader)
            ):

                scaler.unscale_(
                    optimizer
                )

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    1.0
                )

                scaler.step(
                    optimizer
                )

                scaler.update()

                optimizer.zero_grad()

                scheduler.step()

        # validation
        if dataset_name == "SIED-Thai":

            val_y, val_prob = predict_multilabel(
                model,
                val_loader
            )

            val_thresholds = tune_multilabel_thresholds(
                val_y,
                val_prob
            )

            val_metrics, _ = multilabel_metrics(
                val_y,
                val_prob,
                val_thresholds
            )

        else:

            val_y, val_pred = predict_single_label(
                model,
                val_loader
            )

            val_metrics, _ = single_label_metrics(
                val_y,
                val_pred
            )

        val_macro = (
            val_metrics["macro_f1"]
        )

        print(
            f"Epoch {epoch+1:2d}/{EPOCHS} | "
            f"loss={total_loss/len(train_loader):.4f} | "
            f"val_micro={val_metrics['micro_f1']:.4f} | "
            f"val_macro={val_macro:.4f}"
        )

        if val_macro > best_val_macro_f1:

            best_val_macro_f1 = val_macro

            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

    # load best validation epoch
    model.load_state_dict(
        best_state
    )

    model.to(DEVICE)

    if dataset_name == "SIED-Thai":

        val_y, val_prob = predict_multilabel(
            model,
            val_loader
        )

        thresholds = tune_multilabel_thresholds(
            val_y,
            val_prob
        )

        test_y, test_prob = predict_multilabel(
            model,
            test_loader
        )

        overall, per_label = multilabel_metrics(
            test_y,
            test_prob,
            thresholds
        )

    else:

        thresholds = None

        test_y, test_pred = predict_single_label(
            model,
            test_loader
        )

        overall, per_label = single_label_metrics(
            test_y,
            test_pred
        )

    return (
        model,
        tokenizer,
        overall,
        per_label,
        thresholds,
        best_val_macro_f1
    )

In [ ]:
# =========================================================
# RUN ALL — XLM-R + mBERT × 2 EMOTION DATASETS
# =========================================================

all_overall = []
all_per_category = []

experiments = []

if RUN_SIED:
    for model_display, model_name in MODELS.items():
        experiments.append(
            (
                model_display,
                model_name,
                "SIED-Thai"
            )
        )

if RUN_THAI_DEPRESSION:
    for model_display, model_name in MODELS.items():
        experiments.append(
            (
                model_display,
                model_name,
                "Thai-Depression"
            )
        )


for exp_idx, (
    model_display,
    model_name,
    dataset_name
) in enumerate(
    experiments,
    start=1
):

    print("\n" + "=" * 100)
    print(
        f"EXPERIMENT {exp_idx}/{len(experiments)}"
    )
    print(
        f"MODEL   : {model_display} ({model_name})"
    )
    print(
        f"DATASET : {dataset_name}"
    )
    print("=" * 100)

    (
        model,
        tokenizer,
        overall,
        per_category,
        thresholds,
        best_val_macro
    ) = train_standalone(
        model_name,
        dataset_name
    )

    overall_row = {
        "dataset":
            dataset_name,

        "model":
            model_display,

        "model_id":
            model_name,

        "seed":
            SEED,

        "epochs":
            EPOCHS,

        "best_val_macro_f1":
            best_val_macro,

        **overall
    }

    all_overall.append(
        overall_row
    )

    temp = per_category.copy()

    temp.insert(
        0,
        "model",
        model_display
    )

    temp.insert(
        0,
        "dataset",
        dataset_name
    )

    all_per_category.append(
        temp
    )

    print("\nOVERALL TEST METRICS")

    for key, value in overall.items():
        print(
            f"{key:20s}: {value:.6f}"
        )

    print("\nPER-CATEGORY TEST METRICS")

    display(
        temp[
            [
                "dataset",
                "model",
                "label",
                "precision",
                "recall",
                "f1",
                "support",
            ]
            + (
                ["threshold"]
                if "threshold" in temp.columns
                else []
            )
        ]
    )

    if SAVE_MODELS:

        safe_model = (
            model_display
            .replace("/", "_")
            .replace(" ", "_")
        )

        safe_dataset = (
            dataset_name
            .replace("/", "_")
            .replace(" ", "_")
        )

        save_dir = os.path.join(
            OUTPUT_DIR,
            f"{safe_model}_{safe_dataset}_standalone"
        )

        model.save_pretrained(
            save_dir
        )

        tokenizer.save_pretrained(
            save_dir
        )

        print(
            "Saved model:",
            save_dir
        )

    del model, tokenizer

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


overall_df = pd.DataFrame(
    all_overall
)

per_category_df = pd.concat(
    all_per_category,
    ignore_index=True
)

print("\n" + "=" * 100)
print("ALL OVERALL RESULTS")
print("=" * 100)

display(
    overall_df
)


EXPERIMENT 1/4
MODEL   : XLM-R (xlm-roberta-base)
DATASET : SIED-Thai


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Thai-Depression | Epoch 5/5:   0%|          | 0/1563 [00:00<?, ?it/s]

Epoch  5/5 | loss=0.3122 | val_micro=0.8030 | val_macro=0.8022

OVERALL TEST METRICS
accuracy            : 0.773381
micro_precision     : 0.773381
micro_recall        : 0.773381
micro_f1            : 0.773381
macro_precision     : 0.776573
macro_recall        : 0.772119
macro_f1            : 0.772095
weighted_f1         : 0.772464

PER-CATEGORY TEST METRICS


,dataset,model,label,precision,recall,f1,support
0,Thai-Depression,mBERT,no_depression,0.801376,0.713644,0.754970,2448
1,Thai-Depression,mBERT,depression,0.751771,0.830595,0.789219,2556



ALL OVERALL RESULTS


,dataset,model,model_id,seed,epochs,best_val_macro_f1,accuracy,micro_precision,micro_recall,micro_f1,macro_precision,macro_recall,macro_f1,weighted_f1,hamming_loss
0,SIED-Thai,XLM-R,xlm-roberta-base,42,5,0.485632,0.311111,0.505762,0.814433,0.624013,0.388733,0.627241,0.448033,0.695721,0.220370
1,SIED-Thai,mBERT,bert-base-multilingual-cased,42,5,0.379194,0.030556,0.349794,0.876289,0.500000,0.270859,0.664637,0.340588,0.655424,0.393519
2,Thai-Depression,XLM-R,xlm-roberta-base,42,5,0.814354,0.807754,0.807754,0.807754,0.807754,0.807755,0.807517,0.807599,0.807717,NaN
3,Thai-Depression,mBERT,bert-base-multilingual-cased,42,5,0.802169,0.773381,0.773381,0.773381,0.773381,0.776573,0.772119,0.772095,0.772464,NaN


In [ ]:
# =========================================================
# PAPER-READY OVERALL COMPARISON
# =========================================================

paper_overall_cols = [
    "dataset",
    "model",
    "accuracy",
    "micro_f1",
    "macro_f1",
]

paper_overall_df = overall_df[
    paper_overall_cols
].copy()

paper_overall_df = paper_overall_df.sort_values(
    ["dataset", "model"]
).reset_index(drop=True)

print("OVERALL COMPARISON")
display(
    paper_overall_df
)

OVERALL COMPARISON


,dataset,model,accuracy,micro_f1,macro_f1
0,SIED-Thai,XLM-R,0.311111,0.624013,0.448033
1,SIED-Thai,mBERT,0.030556,0.500000,0.340588
2,Thai-Depression,XLM-R,0.807754,0.807754,0.807599
3,Thai-Depression,mBERT,0.773381,0.773381,0.772095


In [ ]:
# =========================================================
# PAPER-READY PER-CATEGORY COMPARISON
# Precision / Recall / F1-score / Support
# =========================================================

paper_per_category_df = (
    per_category_df[
        [
            "dataset",
            "label",
            "model",
            "precision",
            "recall",
            "f1",
            "support",
        ]
    ]
    .copy()
)

paper_per_category_df = (
    paper_per_category_df
    .sort_values(
        ["dataset", "label", "model"]
    )
    .reset_index(drop=True)
)

print("PER-CATEGORY COMPARISON")
display(
    paper_per_category_df
)

PER-CATEGORY COMPARISON


,dataset,label,model,precision,recall,f1,support
0,SIED-Thai,Anger,XLM-R,0.272727,0.675000,0.388489,40
1,SIED-Thai,Anger,mBERT,0.115607,1.000000,0.207254,40
2,SIED-Thai,Disgust,XLM-R,0.316129,0.731343,0.441441,67
3,SIED-Thai,Disgust,mBERT,0.222222,0.865672,0.353659,67
4,SIED-Thai,Fear,XLM-R,0.266667,0.476190,0.341880,42
5,SIED-Thai,Fear,mBERT,0.184783,0.404762,0.253731,42
6,SIED-Thai,Happiness,XLM-R,0.555556,0.434783,0.487805,23
7,SIED-Thai,Happiness,mBERT,0.217391,0.217391,0.217391,23
8,SIED-Thai,Sadness,XLM-R,0.841317,0.946128,0.890650,297
9,SIED-Thai,Sadness,mBERT,0.825000,1.000000,0.904110,297


In [ ]:
# =========================================================
# TABLE FORMAT LIKE PAPER:
# one table per SIED emotion
# =========================================================

if RUN_SIED:

    for label in SIED_LABELS:

        print("\n" + "=" * 70)
        print(label.upper())
        print("=" * 70)

        table = (
            paper_per_category_df[
                (
                    paper_per_category_df["dataset"]
                    == "SIED-Thai"
                )
                &
                (
                    paper_per_category_df["label"]
                    == label
                )
            ][
                [
                    "model",
                    "precision",
                    "recall",
                    "f1",
                    "support",
                ]
            ]
            .rename(
                columns={
                    "model": "Model",
                    "precision": "Precision",
                    "recall": "Recall",
                    "f1": "F1-score",
                    "support": "Support",
                }
            )
            .reset_index(drop=True)
        )

        display(table)


HAPPINESS


,Model,Precision,Recall,F1-score,Support
0,XLM-R,0.555556,0.434783,0.487805,23
1,mBERT,0.217391,0.217391,0.217391,23



SADNESS


,Model,Precision,Recall,F1-score,Support
0,XLM-R,0.841317,0.946128,0.89065,297
1,mBERT,0.825000,1.000000,0.90411,297



ANGER


...

,Model,Precision,Recall,F1-score,Support
0,XLM-R,0.316129,0.731343,0.441441,67
1,mBERT,0.222222,0.865672,0.353659,67



SURPRISE


,Model,Precision,Recall,F1-score,Support
0,XLM-R,0.08000,0.5,0.137931,16
1,mBERT,0.06015,0.5,0.107383,16



FEAR


,Model,Precision,Recall,F1-score,Support
0,XLM-R,0.266667,0.476190,0.341880,42
1,mBERT,0.184783,0.404762,0.253731,42


In [ ]:
# =========================================================
# THAI-DEPRESSION TABLE
# =========================================================

if RUN_THAI_DEPRESSION:

    for label in DEPRESSION_LABELS:

        print("\n" + "=" * 70)
        print(label.upper())
        print("=" * 70)

        table = (
            paper_per_category_df[
                (
                    paper_per_category_df["dataset"]
                    == "Thai-Depression"
                )
                &
                (
                    paper_per_category_df["label"]
                    == label
                )
            ][
                [
                    "model",
                    "precision",
                    "recall",
                    "f1",
                    "support",
                ]
            ]
            .rename(
                columns={
                    "model": "Model",
                    "precision": "Precision",
                    "recall": "Recall",
                    "f1": "F1-score",
                    "support": "Support",
                }
            )
            .reset_index(drop=True)
        )

        display(table)


NO_DEPRESSION


,Model,Precision,Recall,F1-score,Support
0,XLM-R,0.807788,0.796569,0.802139,2448
1,mBERT,0.801376,0.713644,0.754970,2448



DEPRESSION


,Model,Precision,Recall,F1-score,Support
0,XLM-R,0.807722,0.818466,0.813059,2556
1,mBERT,0.751771,0.830595,0.789219,2556


In [ ]:
# =========================================================
# EXPORT CSV
# =========================================================

paths = {
    "overall_full":
        os.path.join(
            OUTPUT_DIR,
            "xlmr_mbert_standalone_overall_full.csv"
        ),

    "overall_paper":
        os.path.join(
            OUTPUT_DIR,
            "xlmr_mbert_standalone_overall_paper.csv"
        ),

    "per_category_full":
        os.path.join(
            OUTPUT_DIR,
            "xlmr_mbert_standalone_per_category_full.csv"
        ),

    "per_category_paper":
        os.path.join(
            OUTPUT_DIR,
            "xlmr_mbert_standalone_per_category_paper.csv"
        ),
}

overall_df.to_csv(
    paths["overall_full"],
    index=False,
    encoding="utf-8-sig"
)

paper_overall_df.to_csv(
    paths["overall_paper"],
    index=False,
    encoding="utf-8-sig"
)

per_category_df.to_csv(
    paths["per_category_full"],
    index=False,
    encoding="utf-8-sig"
)

paper_per_category_df.to_csv(
    paths["per_category_paper"],
    index=False,
    encoding="utf-8-sig"
)

print("EXPORTED FILES")

for key, value in paths.items():
    print(
        f"{key:20s} -> {value}"
    )

EXPORTED FILES
overall_full         -> /content/xlmr_mbert_standalone_overall_full.csv
overall_paper        -> /content/xlmr_mbert_standalone_overall_paper.csv
per_category_full    -> /content/xlmr_mbert_standalone_per_category_full.csv
per_category_paper   -> /content/xlmr_mbert_standalone_per_category_paper.csv
